# Laikipia farm digitisation: train and retrain the model

Run the cells **top to bottom**. Each step says what it does and what to look at.

```
fetch Sentinel-2  ->  labels from YOUR polygons  ->  train U-Net  ->  evaluate on unseen blocks
                                                                         |
   corrected polygons from Google Earth Pro  <-  predict new area  <-----+
          \_____________________  retrain (fine-tune)  ___________________/
```

Parts: **A** setup and data, **B** train, **C** evaluate, **D** predict, **E** retrain with new images.

## A. Setup

In [ ]:
import os, sys, json, shutil
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)          # run from the project folder
sys.path.insert(0, str(Path.cwd()))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import rasterio, geopandas as gpd
from rasterio.enums import Resampling

from farmmap.config import load_config
from farmmap.model import pick_device

cfg = load_config("config.yaml")
print("Project folder:", cfg.root)
print("Compute device:", pick_device(cfg.train.device), "(cpu is fine, just slower)")

### A1. Your reference polygons
These are the polygons you digitised. They become the training labels, so their quality sets the ceiling for the model.

In [ ]:
from farmmap.reference import load_reference

ref = load_reference(cfg)
area_ha = ref.area / 1e4
print(f"{len(ref)} reference polygons, total {area_ha.sum():,.0f} ha")
print(area_ha.describe().round(1).to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ref.plot(ax=ax[0], facecolor="none", edgecolor="k", linewidth=0.5)
ax[0].set_title("Reference polygons")
ax[1].hist(np.log10(area_ha), bins=30)
ax[1].set_xlabel("log10(area, ha)"); ax[1].set_title("Farm size distribution")
plt.show()

### A2. Optional: change settings for this run
Everything lives in `config.yaml`; override here for a quick experiment. For a first run keep the pilot `bbox` small.

In [ ]:
# cfg.imagery.bbox = [36.70, 0.10, 37.00, 0.40]   # [min_lon, min_lat, max_lon, max_lat]; None = whole layer
# cfg.train.epochs = 10                           # shorter run to test the setup
# cfg.model.encoder = "resnet34"                  # larger model, slower on CPU
print("bbox:", cfg.imagery.bbox, "| epochs:", cfg.train.epochs, "| encoder:", cfg.model.encoder)

### A3. How big is the download?

In [ ]:
from farmmap.imagery import aoi_bounds, estimate
print(estimate(cfg, aoi_bounds(cfg)))

### A4. Extract the satellite images
Builds a cloud-free median composite of the least cloudy Sentinel-2 scenes from the free Planetary Computer.
**Resumable:** if the connection drops, run the cell again and finished chunks are skipped.

In [ ]:
from farmmap.imagery import fetch_composite
img_path = fetch_composite(cfg)
print(img_path)

In [ ]:
def rgb_preview(path, max_px=1200):
    with rasterio.open(path) as src:
        s = max(1, max(src.width, src.height) // max_px)
        d = src.read([3, 2, 1], out_shape=(3, src.height // s, src.width // s),
                     resampling=Resampling.average).astype("float32")
        extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)
        crs = src.crs
    valid = (d > 0).all(0)
    lo, hi = np.percentile(d[:, valid], (2, 98)) if valid.any() else (0, 1)
    rgb = np.clip((d - lo) / (hi - lo + 1e-6), 0, 1).transpose(1, 2, 0)
    rgb[~valid] = 1
    return rgb, extent, crs

rgb, extent, crs = rgb_preview(img_path)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb, extent=extent)
ref.to_crs(crs).boundary.plot(ax=ax, color="yellow", linewidth=0.7)
ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
ax.set_title("Composite with your polygons. Do boundaries match visible features?")
plt.show()

### A5. Build training labels from your polygons
Each pixel becomes **0 background / 1 farm interior / 2 farm boundary** (255 = ignore: no imagery or far from any polygon).
This step is repeated whenever you add polygons or images (section E).

In [ ]:
from farmmap.pipeline import prepare_training_data
scenes = prepare_training_data(cfg)
for s in scenes: print(s["name"], "->", s["labels"].name)

In [ ]:
# Look at a sample: image and labels
s0 = scenes[0]
with rasterio.open(s0["imagery"]) as si, rasterio.open(s0["labels"]) as sl:
    r, c = si.height // 2 - 256, si.width // 2 - 256
    win = rasterio.windows.Window(max(c, 0), max(r, 0), 512, 512)
    im = si.read([3, 2, 1], window=win).astype("float32"); lb = sl.read(1, window=win)
im = np.clip(im / max(np.percentile(im[im > 0], 98), 1), 0, 1).transpose(1, 2, 0) if (im > 0).any() else im.transpose(1, 2, 0)
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(im); ax[0].set_title("Image (R,G,B)")
cmap = plt.matplotlib.colors.ListedColormap(["#444444", "#4caf50", "#ffeb3b"])
ax[1].imshow(np.where(lb == 255, np.nan, lb), cmap=cmap, vmin=0, vmax=2, interpolation="nearest")
ax[1].set_title("Labels: grey bg / green interior / yellow boundary"); plt.show()

### A6. Train / validation / test split
Whole 5 km blocks are held out, so the test score measures performance on **places the model has never seen**.
The assignment is saved in `data/labels/split.json` and never changes, so later models are scored on the same test blocks.

In [ ]:
from farmmap.data import load_or_create_split, split_blocks
split = load_or_create_split(cfg, scenes)
for name in ("train", "val", "test"):
    print(name, len(split_blocks(split, name)), "blocks")

## B. Train the model

**Architecture:** a U-Net (encoder `resnet18`, ImageNet-pretrained) that reads the 6 Sentinel-2 bands + NDVI and predicts 3 classes per pixel.
Predicting the **boundary** as its own class is what lets us cut touching farms apart into separate polygons later.

**Loss:** class-weighted cross-entropy + Dice (boundary pixels are rare, so they are up-weighted).

In [ ]:
from farmmap.data import ensure_norm, n_channels
from farmmap.model import build_model

norm = ensure_norm(cfg, scenes[0]["imagery"])
net = build_model(cfg.model.encoder, n_channels(norm), encoder_weights=None)
print(f"input channels: {n_channels(norm)} | parameters: {sum(p.numel() for p in net.parameters())/1e6:.1f} M")
del net

### B1. Run training
Prints one line per epoch and saves the **best** epoch (by mean IoU of interior + boundary on validation blocks) to
`data/models/unet_best.pt`. On a CPU laptop expect a few minutes per epoch with the default settings.
Training stops early when validation stops improving. Interrupt the kernel to stop; the best checkpoint so far is kept.

In [ ]:
from farmmap.train import train
result = train(cfg, scenes)          # from scratch
print("Best epoch:", result["best"].get("epoch"), "| score:", round(result["best"].get("score", 0), 3))
print("Checkpoint:", result["checkpoint"])

In [ ]:
h = pd.read_csv(result["history_csv"])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h.epoch, h.iou_interior, label="interior IoU"); ax[1].plot(h.epoch, h.iou_boundary, label="boundary IoU")
ax[1].set_title("Validation IoU"); ax[1].set_xlabel("epoch"); ax[1].legend(); plt.show()

## C. Evaluate on unseen test blocks
Interpretation guide:
- **iou_interior**: how well farm *areas* are found.
- **boundary_f1** (within +/-20 m): how well farm *edges* are found. Expect this to be the weak point wherever no fence, road or land-use change exists.
- **farm_recall / farm_precision** (IoU >= 0.5): the fraction of your polygons recovered, and the fraction of predicted polygons that are right.

In [ ]:
from farmmap.evaluate import evaluate
metrics_v1 = evaluate(cfg, scenes)
pd.Series(metrics_v1).to_frame("v1")

In [ ]:
# Visual check on one test block: image, probabilities, predicted vs your polygons
from farmmap.data import block_window
from farmmap.predict import predict_raster
from farmmap.polygonize import polygonize_probs
import tempfile

test_blocks = split_blocks(split, "test")
b = test_blocks[0]
scene = next(s for s in scenes if s["name"] == b["scene"])
win = block_window(b)
with tempfile.TemporaryDirectory() as tmp:
    pr = predict_raster(cfg, scene["imagery"], None, Path(tmp) / "p.tif", [win], progress=lambda *_: None)
    with rasterio.open(pr) as sp, rasterio.open(scene["imagery"]) as si:
        probs = sp.read(window=win) / 255.0
        rgbw = si.read([3, 2, 1], window=win).astype("float32")
        tr = si.window_transform(win)
    polys = polygonize_probs(cfg, pr, progress=lambda *_: None)
rgbw = np.clip(rgbw / max(np.percentile(rgbw[rgbw > 0], 98), 1), 0, 1).transpose(1, 2, 0)
ext = (tr.c, tr.c + tr.a * win.width, tr.f + tr.e * win.height, tr.f)
fig, ax = plt.subplots(1, 3, figsize=(16, 5.5))
ax[0].imshow(rgbw, extent=ext); ax[0].set_title("Image")
ax[1].imshow(np.dstack([probs[2], probs[1], probs[0] * 0]), extent=ext); ax[1].set_title("P(boundary)=red, P(interior)=green")
ax[2].imshow(rgbw, extent=ext)
if len(polys):
    polys.boundary.plot(ax=ax[2], color="cyan", linewidth=1.2)
load_reference(cfg, crs=polys.crs).boundary.plot(ax=ax[2], color="yellow", linewidth=1.2, linestyle="--")
ax[2].set_xlim(ext[0], ext[1]); ax[2].set_ylim(ext[2], ext[3]); ax[2].set_title("cyan = model, yellow = yours")
plt.show()

## D. Use the model: image -> polygons -> Google Earth Pro

Writes a `.kmz` (open in Google Earth Pro), a `.gpkg` (QGIS) and the probability raster to `outputs/<name>/`.

In [ ]:
from farmmap.pipeline import predict_area
res = predict_area(cfg, scenes[0]["imagery"], "pilot")
print(len(res["frame"]), "farms")
print("Open in Google Earth Pro:", res["kmz"])
res["frame"].head()

## E. Retrain with new images to get a better model

1. **Extract images** for areas where the model is weak or that you want to add (cell E1).
2. **Predict** there and open the KMZ in Google Earth Pro on top of the imagery.
3. **Correct / add polygons** exactly as you already do. Save as KML or KMZ into `data/labels/manual/`.
   - Digitise *whole* farms and be consistent about what counts as a boundary.
   - A polygon you draw **replaces** any earlier polygon it covers by more than half.
   - Areas with no polygons nearby are ignored, not treated as "no farms".
4. **Fine-tune** from the current model on all scenes (cell E2). It keeps what it learned and adapts to your corrections.
5. **Compare** old and new on the *same* test blocks and keep the new model only if it is better (cells E3 and E4).

In [ ]:
# E1. Extract a new area as an extra training scene (any bbox [min_lon, min_lat, max_lon, max_lat]
#     or aoi_path="my_area.kml"). It is saved in data/imagery/ and automatically used for training.
from farmmap.imagery import fetch_composite

NEW_NAME = "area_2"
NEW_BBOX = [36.95, -0.05, 37.10, 0.10]
new_img = fetch_composite(cfg, out_path=cfg.path(cfg.imagery.dir) / f"{NEW_NAME}.tif", bbox=NEW_BBOX)
new_pred = predict_area(cfg, new_img, NEW_NAME)
print("Review and correct in Google Earth Pro:", new_pred["kmz"])
print("Then save your corrected polygons into:", cfg.path("data/labels/manual"))

In [ ]:
# E2. After saving corrections into data/labels/manual/ : rebuild labels for ALL scenes and fine-tune.
scenes = prepare_training_data(cfg)

v1 = cfg.path(cfg.model.checkpoint)          # current model
v2 = cfg.path("data/models/unet_v2.pt")      # new model (kept separate until we compare)
result2 = train(cfg, scenes, init_checkpoint=v1, out_checkpoint=v2, epochs=15)
print("Best epoch:", result2["best"].get("epoch"), "| score:", round(result2["best"].get("score", 0), 3))

In [ ]:
# E3. Compare v1 and v2 on the same held-out test blocks
m1 = evaluate(cfg, scenes, checkpoint=v1)
m2 = evaluate(cfg, scenes, checkpoint=v2)
table = pd.DataFrame({"v1": m1, "v2": m2}).drop(["split", "checkpoint"])
table

In [ ]:
# E4. Promote v2 only if it is better. The old model is backed up first.
PROMOTE = False        # <- set to True after looking at the table above
if PROMOTE:
    shutil.copy(v1, v1.with_name("unet_v1_backup.pt"))
    shutil.copy(v2, v1)
    print("unet_best.pt is now the retrained model (previous saved as unet_v1_backup.pt)")
else:
    print("Not promoted. Set PROMOTE = True to adopt the retrained model.")

**Tips for getting a better model**

- More *varied* corrected areas beat more polygons in one place: include different land uses, farm sizes and terrain.
- If boundary IoU is low but interior IoU is good, add polygons where boundaries are clearly visible (fences, roads, rivers) and check that your polygons follow those features.
- Try `cfg.model.encoder = "resnet34"` and more `cfg.train.epochs` if you have time or a GPU.
- For small plots (< 5 ha), Sentinel-2 at 10 m is too coarse. Use higher-resolution imagery you are licensed to use (`imagery.source: local`).